# ✈️ Intelligent Alert Prioritization (Goal-Based Agent)
CrewAI + Groq + Llama 3.1 8B Instant

Goal: Reduce pilot overload by combining correlated alerts intelligently.

In [ ]:
!pip install crewai langchain langchain-groq gradio

In [ ]:
import os
from crewai import Agent, Task, Crew
from langchain_groq import ChatGroq
import gradio as gr

In [ ]:
# Set your Groq API key
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)

In [ ]:
alert_agent = Agent(
    role="Aerospace Safety Monitoring Agent",
    goal="Prioritize and combine alerts to reduce pilot overload while ensuring safety",
    backstory="""
    You are an advanced AI system used in modern aircraft to monitor engine health.
    Your job is to intelligently combine multiple warning signals into meaningful alerts.
    Avoid overwhelming the pilot with redundant alerts.
    """,
    llm=llm,
    verbose=True
)

In [ ]:
def create_task(temp, vibration, temp_threshold=800, vibration_threshold=70):

    task_description = f"""
    Analyze the following engine parameters:

    - Temperature: {temp} °C (Threshold: {temp_threshold})
    - Vibration: {vibration} Hz (Threshold: {vibration_threshold})

    Rules:
    1. If BOTH temperature and vibration exceed threshold → COMBINED CRITICAL ALERT
    2. If only one exceeds → SINGLE ALERT
    3. If none exceed → NORMAL

    Goal:
    Reduce pilot overload by combining alerts intelligently.

    Provide:
    - Alert Level (NORMAL / WARNING / CRITICAL)
    - Alert Type (Single / Combined)
    - Explanation
    - Recommended Action
    - Tone: Mission Control AI
    """

    return Task(
        description=task_description,
        agent=alert_agent,
        expected_output="Structured aerospace safety report"
    )

In [ ]:
def run_alert_system(temp, vibration):
    task = create_task(temp, vibration)

    crew = Crew(
        agents=[alert_agent],
        tasks=[task],
        verbose=True
    )

    result = crew.kickoff()
    return result

In [ ]:
def launch_ui():
    with gr.Blocks(theme=gr.themes.Soft()) as demo:

        gr.HTML("""
        <div style="text-align:center; padding:20px;">
            <h1 style="color:#0B3D91;">✈️ Intelligent Alert Prioritization</h1>
            <h3 style="color:gray;">Collins Aerospace AI Safety System</h3>
            <marquee style="color:#0B3D91; font-weight:bold;">
                --- Multi-Sensor Fusion | Pilot Load Reduction | AI Safety Intelligence ---
            </marquee>
        </div>
        """)

        with gr.Row():
            temp_input = gr.Slider(0, 2000, value=500, label="Engine Temperature (°C)")
            vibration_input = gr.Slider(0, 200, value=50, label="Engine Vibration (Hz)")

        output_box = gr.Textbox(label="AI Prioritized Alert Report", lines=15)

        run_btn = gr.Button("Run Alert Analysis")

        run_btn.click(
            fn=run_alert_system,
            inputs=[temp_input, vibration_input],
            outputs=output_box
        )

    return demo

app = launch_ui()
app.launch()